<a href="https://colab.research.google.com/github/sridevi-t/project_using_dataset/blob/main/mbti11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from google.colab import files
uploaded = files.upload()

Saving mbti_1.xlsx.csv to mbti_1.xlsx (1).csv


In [7]:
from google.colab import drive
drive.mount('/content/mbti_1.xlsx.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
uploaded.keys()

dict_keys(['mbti_1.xlsx (1).csv'])

In [11]:
import pandas as pd

df = pd.read_csv("mbti_1.xlsx (1).csv")
df.head()

,type,posts
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...
1,ENTP,'I'm finding the lack of me in these posts ver...
2,INTP,'Good one _____ https://www.youtube.com/wat...
3,INTJ,"'Dear INTP, I enjoyed our conversation the o..."
4,ENTJ,'You're fired.|||That's another silly misconce...


In [12]:
print(df.shape)
print(df.columns)

(8675, 2)
Index(['type', 'posts'], dtype='object')


In [13]:
import re
import nltk
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [14]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df['cleaned_posts'] = df['posts'].apply(clean_text)
df.head()

,type,posts,cleaned_posts
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...,intj moments sportscenter top ten plays pranks...
1,ENTP,'I'm finding the lack of me in these posts ver...,im finding lack posts alarmingsex boring posit...
2,INTP,'Good one _____ https://www.youtube.com/wat...,good one course say know thats blessing cursed...
3,INTJ,"'Dear INTP, I enjoyed our conversation the o...",dear intp enjoyed conversation day esoteric ga...
4,ENTJ,'You're fired.|||That's another silly misconce...,youre firedthats another silly misconception a...


In [15]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['type'])

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_posts'],
    df['label'],
    test_size=0.2,
    random_state=42
)
vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [20]:
model = LogisticRegression(max_iter=200)
model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=200)

In [21]:
y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.6334293948126801
              precision    recall  f1-score   support

           0       0.50      0.10      0.16        41
           1       0.71      0.58      0.64       125
           2       0.79      0.34      0.48        44
           3       0.68      0.52      0.59       135
           4       0.00      0.00      0.00         7
           5       0.00      0.00      0.00         8
           6       0.00      0.00      0.00         7
           7       1.00      0.13      0.24        15
           8       0.63      0.69      0.66       288
           9       0.60      0.86      0.70       370
          10       0.61      0.70      0.65       193
          11       0.63      0.78      0.70       293
          12       0.91      0.22      0.36        45
          13       0.69      0.21      0.32        53
          14       0.71      0.23      0.34        44
          15       0.74      0.37      0.50        67

    accuracy                           0.63      17

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [22]:
def predict_mbti(text):
    text = clean_text(text)
    vector = vectorizer.transform([text])
    pred = model.predict(vector)
    return le.inverse_transform(pred)[0]

# try it
text = input("Enter text: ")
print("Predicted MBTI:", predict_mbti(text))

Enter text: i feel so insecure today
Predicted MBTI: INFP
